In [1]:
!pip install datasets

In [11]:
high_acc_styles = [
    'Color_Field_Painting',
    'Early_Renaissance',
    'Naive_Art_Primitivism',
    'Rococo',
    'Northern_Renaissance',
    'Baroque',
    'Art_Nouveau',
    'Romanticism',
    'Impressionism',
    'Realism',
    'Ukiyo_e'
]

In [32]:
#download style imgs

import os
from datasets import load_dataset, Dataset, disable_caching
import itertools, datasets
import nest_asyncio; nest_asyncio.apply()

stream_ds = load_dataset(
    "huggan/wikiart",
    split="train",
    streaming=True
)
stream_ds = stream_ds.shuffle(seed=22, buffer_size=5000)
num_threads = num_threads = min(32, (os.cpu_count() or 1) + 4)
stream_ds = stream_ds.decode(num_threads=num_threads)
first_300 = stream_ds.take(500)
style_feature = stream_ds.features["style"]
idx_to_style = {i: style_feature.int2str(i) for i in range(style_feature.num_classes)}
style_to_idx = style_feature.str2int

style_count = {style: 0 for style in idx_to_style.values()}

for example in first_300:
    style = idx_to_style[example["style"]]
    style_count[style] += 1

final_subset = []
style_count_final = {style: 0 for style in idx_to_style.values()}

print("Style counts in the subset of 50 images:")
for style, count in style_count.items():
    print(f"{style}: {count}")

for img in first_300:
    if len(final_subset) >= 60:
        break
    style = idx_to_style[img["style"]]

    if style in high_acc_styles:
      if style_count[style] >= 6 and style_count_final[style] <= 5:
          final_subset.append(img)
          style_count_final[style] += 1
    elif len(final_subset)>=30:
      if style_count[style] >= 3 and style_count_final[style] <= 2:
          final_subset.append(img)
          style_count_final[style] += 1



Resolving data files:   0%|          | 0/72 [00:00<?, ?it/s]

Style counts in the subset of 50 images:
Abstract_Expressionism: 5
Action_painting: 0
Analytical_Cubism: 2
Art_Nouveau: 74
Baroque: 17
Color_Field_Painting: 0
Contemporary_Realism: 0
Cubism: 12
Early_Renaissance: 1
Expressionism: 26
Fauvism: 1
High_Renaissance: 6
Impressionism: 125
Mannerism_Late_Renaissance: 2
Minimalism: 2
Naive_Art_Primitivism: 18
New_Realism: 0
Northern_Renaissance: 17
Pointillism: 1
Pop_Art: 8
Post_Impressionism: 33
Realism: 71
Rococo: 5
Romanticism: 32
Symbolism: 37
Synthetic_Cubism: 0
Ukiyo_e: 5


In [33]:
images_final_subset = [img["image"] for img in final_subset]
print(final_subset[0])
print(idx_to_style[final_subset[0]["style"]])

{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=1382x1865 at 0x79A5F878B140>, 'artist': 25, 'genre': 8, 'style': 17}
Northern_Renaissance


In [34]:
idx_to_style[final_subset[0]["style"]]

'Northern_Renaissance'

In [36]:
out = "s_def"
counts = {}

for img, rec in zip(images_final_subset, final_subset):
    style = idx_to_style[rec["style"]]
    i = counts.get(style, 0)
    img.save(out + f"/{style}_{i}.jpg", "JPEG")
    counts[style] = i + 1


In [37]:
import shutil

# source folder and output zip file name
folder_path = "s_def"
zip_name = "s_def"

shutil.make_archive(zip_name, 'zip', folder_path)


'/content/s_def.zip'